# 权重衰退（Weight Decay）· L2 正则化 — 详解笔记

## 1. 要解决什么毛病：过拟合
- 过拟合：模型为命中训练集每个点（含噪声），曲线剧烈抖动，把噪声当规律 → 测试集全错
- 关键事实：过拟合模型的权重通常很大
  - 权重大 → 函数对输入敏感（输入微动，输出狂跳）→ 容易记住噪声而非规律
- 朴素想法：**强制让权重别太大** → 这就是权重衰退

## 2. 均方范数 = 给权重称"总体积"
- 权重向量 w = (w₁, w₂, …, w_d)
- 范数：衡量这串数字整体有多大的尺子
- 均方范数（平方 L2 范数）：‖w‖² = w₁² + w₂² + … + w_d²
- 选"平方"是因为求导干净（d(‖w‖²)/dw = 2w）

## 3. 核心操作：把"限制"变成损失里的一项
原目标：min L(w)（仅拟合误差）
新目标：
> L'(w) = L(w) + (λ/2)·‖w‖²
- 左边 L(w)：把数据拟合好
- 右边 (λ/2)·‖w‖²：权重越大，罚款越重；λ = 罚款单价

### 一道极简算术（证明权重真的被压小）
设只有一个权重 w，数据项 L(w) = (w − a)²，加惩罚：
> L'(w) = (w − a)² + (λ/2)w²
求导令为 0：2(w − a) + λw = 0 → w = 2a / (2 + λ)
- λ = 0：w = a（原样）
- λ > 0：分母变大 → w 被往 0 拉小

## 4. 几何直觉：两座山谷"拔河"
- 数据项 L(w)：在真实最优 w* 处有个最低谷
- 惩罚项 (λ/2)‖w‖²：以原点 w=0 为最低点的碗，离原点越远地势越高
- 最终最优 = 两地形叠加后的最低点，被惩罚碗从 w* 往 0 方向拉一点；λ 越大拉得越狠

## 5. "衰退"精确落在更新公式里
梯度：∂L'/∂w = ∂L/∂w + λw
更新（η 为学习率）：
> w ← w − η(∂L/∂w + λw) = (1 − ηλ)·w − η·∂L/∂w
- (1 − ηλ) 严格 < 1（λ,η > 0）
- 每步先整体乘一个略小于 1 的因子，把权重往 0 拽一点 → 持续变小 = "衰退"

### 极端情形（数据梯度为 0 时）
> w_new = (1 − ηλ)·w_old  →  w_t = (1 − ηλ)^t · w₀  → 指数衰减到 0

## 6. 硬性限制 vs 柔性限制（墙 vs 斜坡）
- 硬性限制：直接下令 ‖w‖² ≤ θ，超出硬砍回球面（一堵墙，墙外不许去）
  - 缺点：带不等式约束的优化难解；θ 难选
- 柔性限制（权重衰退）：墙外每走一步收"过路费" ∝ ‖w‖²（一道斜坡，越远越贵）
  - 优化器自己权衡"多拟合"值不值得"多交罚款"
- 等价性：拉格朗日乘子证明，每个 θ 都对应一个 λ，使两者最优解完全相同
  - 柔性限制 = 硬约束的连续化、好优化版本 → 即"柔性"

## 7. 一句话收口
加 (λ/2)·‖w‖² → 梯度多 λw → 更新时权重每步 ×(1−ηλ)<1 → 持续衰减变小 → 函数变平滑 → 不再死记噪声 → 抑制过拟合

## 8. 实用要点
- λ 是旋钮：λ 大→压得狠→可能欠拟合；λ 小→接近无正则→可能过拟合（需调）
- 偏置通常不惩罚（只对权重 w 加惩罚）
- 与"学习率衰减"不同：前者惩罚权重幅度，后者调小步长
- AdamW 用解耦形式（单独对 w 乘 1−ηλ），否则正则被动量稀释